# 2주차. 자연어를 구조화된 요청으로

**부제:** Structured output: personal_schedule, group_schedule, todo, reminder, unknown 구조화

## 0. 목표

이번 주에는 사용자 요청을 Pydantic으로 `personal_schedule`, `group_schedule`, `todo`, `reminder`, `unknown` 중 하나로 구조화한다.

성취 기준:

- 자유 문장 응답과 structured output의 차이를 설명할 수 있다.
- `kind`, `personal_schedule`, `group_schedule`, `todo`, `reminder`, `unknown`의 역할을 구분할 수 있다.
- `title`, `date`, `start_time`, `end_time`, `members`, `priority`, `reason`을 검증 가능한 payload로 확인할 수 있다.

![structuredoutput](imgs/Structured_outputs.jpg)

## 1. 준비

로컬 `Jupyter Notebook` 또는 `JupyterLab`에서 repo 루트를 기준으로 실행한다.

모든 실습은 프록시 서버를 통해 모델 API를 호출한다. 프록시 토큰, URL, 모델 설정은 repo 루트의 `.env`에서 읽는다.

처음 읽는 순서: `docs/orientation.md` → 이 노트북 → 기본 구조화 실습 → 카나메이트 확장 예제 순서로 진행한다.


In [ ]:
# 흐름: repo 설정과 공통 helper를 준비한다.
# 2주차의 핵심 결과는 자유 문장이 아니라 structured_response다.
import json
import os
import sys
from pathlib import Path
from typing import Any, Literal


def find_repo_root(start: Path) -> Path:
    # 현재 폴더부터 부모 폴더로 올라가며 repo 루트를 찾는다.
    # 노트북을 notebook/ 안에서 실행해도, repo 루트에서 실행해도 같은 경로를 쓰기 위한 장치다.
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. repo 안에서 노트북을 실행하세요.")


# 앞으로 나오는 상대 경로는 모두 이 repo 루트를 기준으로 맞춘다.
REPO_ROOT = find_repo_root(Path.cwd())

if str(REPO_ROOT) not in sys.path:
    # 노트북이 repo 루트의 설정 파일을 찾을 수 있게 경로를 추가한다.
    sys.path.insert(0, str(REPO_ROOT))
# 파일 저장/DB 생성 위치가 흔들리지 않도록 작업 폴더를 repo 루트로 고정한다.
os.chdir(REPO_ROOT)

from dotenv import dotenv_values, load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# override=True를 사용해 노트북 커널에 남아 있던 이전 환경 변수보다 repo .env를 우선한다.
# 이 노트북에서 쓰는 프록시/모델 설정값은 repo 루트의 .env 파일에서만 읽는다.
ENV_PATH = REPO_ROOT / ".env"
load_dotenv(ENV_PATH, override=True)
ENV_VALUES = dotenv_values(ENV_PATH)


def required_env(name: str) -> str:
    value = (ENV_VALUES.get(name) or "").strip()
    if not value or value == "여기에 api key 입력":
        raise RuntimeError(f"repo 루트 .env 파일에 {name}을 설정한 뒤 다시 실행하세요.")
    return value


PROXY_TOKEN = required_env("PROXY_TOKEN")
PROXY_URL = required_env("PROXY_URL")
EMBEDDING_PROXY_URL = required_env("EMBEDDING_PROXY_URL")
OPENAI_MODEL = required_env("OPENAI_MODEL")
OPENAI_EMBEDDING_MODEL = required_env("OPENAI_EMBEDDING_MODEL")


def make_model(max_tokens: int = 500) -> ChatOpenAI:
    # temperature=0은 같은 입력에서 비슷한 tool 선택/구조화 결과가 나오게 하기 위한 설정이다.
    return ChatOpenAI(
        model=OPENAI_MODEL,
        api_key=PROXY_TOKEN,
        base_url=PROXY_URL,
        temperature=0,
        max_completion_tokens=max_tokens,
    )


def show_json(value: Any) -> None:
    # ensure_ascii=False로 한글 payload를 사람이 읽기 쉬운 그대로 출력한다.
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    # agent 실행 결과에서 마지막 message가 사용자에게 보이는 최종 답변이다.
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    # LangChain message 전체를 그대로 보여주면 복잡하므로, 수업에서 볼 핵심만 뽑는다.
    trace = []
    for message in agent_result.get("messages", []):
        # tool_calls는 모델이 "이 도구를 이 인자로 실행해줘"라고 요청한 기록이다.
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            # type == "tool"인 message는 실제 tool 실행 결과다.
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace


## 2. 개념

오늘의 큰 그림: structured output은 모델 답변을 엑셀이나 앱에 넣기 좋은 양식으로 받는 방법이다. "그럴듯한 문장" 대신 정해진 필드가 있는 객체를 받아야 다음 코드가 안정적으로 동작한다.

반드시 이해할 것:

- `response_format`은 모델 출력 모양을 Pydantic 모델에 맞춘다.
- `kind`는 요청 종류를 알려주는 라벨이다.
- `reminder`처럼 새 유형이 추가되면 schema와 검증 기준도 함께 바뀐다.

막혔을 때 볼 trace: 이번 주의 핵심은 tool trace보다 `structured_response.model_dump()`의 `kind`와 세부 필드다.


### 지원
- LLM API에서 지원하기도 한다.
[OpenAI의 structured output](https://developers.openai.com/api/docs/guides/structured-outputs)

## 3. 기본 개념 실습

가장 작은 흐름은 "자연어 요청 → Pydantic response_format → 구조화 객체"이다. 먼저 일정, 할 일, 알 수 없는 요청만 분류한다.


In [ ]:
# 흐름: 일정과 할 일 요청을 앱에서 쓰기 좋은 Pydantic 구조로 정의한다.
# Field 설명은 모델이 어떤 형식으로 채워야 하는지 알려주는 힌트다.
class ScheduleCreate(BaseModel):
    # BaseModel의 각 필드는 앱에서 사용할 데이터 key가 된다.
    title: str = Field(description="일정 제목")
    date: str = Field(description="YYYY-MM-DD")
    start_time: str = Field(description="HH:MM")
    attendees: list[str] = Field(default_factory=list)

class TodoCreate(BaseModel):
    # due_date는 없을 수도 있으므로 str | None으로 둔다.
    title: str
    due_date: str | None = Field(default=None, description="YYYY-MM-DD")
    priority: Literal["low", "medium", "high"] = "medium"

class ExtractionResult(BaseModel):
    # kind를 먼저 보고 schedule/todo/question 중 어떤 필드를 읽을지 결정한다.
    kind: Literal["schedule", "todo", "unknown"]
    schedule: ScheduleCreate | None = None
    todo: TodoCreate | None = None
    question: str | None = None

extract_agent = create_agent(
    model=make_model(),
    tools=[],
    # response_format을 지정하면 모델 답변이 이 Pydantic 모델로 검증된다.
    response_format=ExtractionResult,
    system_prompt="오늘은 2026-04-23이다. 사용자 요청을 schedule, todo, unknown 중 하나로 구조화한다.",
)

student_request = "금요일까지 보고서 제출 할 일 high"
result = extract_agent.invoke({"messages": [{"role": "user", "content": student_request}]})
# structured_response에는 문자열 답변이 아니라 ExtractionResult 객체가 들어 있다.
structured_response = result["structured_response"]
show_json(structured_response.model_dump())


{
  "kind": "todo",
  "schedule": null,
  "todo": {
    "title": "보고서 제출",
    "due_date": "2026-04-24",
    "priority": "high"
  },
  "question": null
}


## 4. 카나메이트 확장 예제

기본 `personal_schedule/group_schedule/todo` 구조에 `reminder` 타입을 추가한다. 이 확장 agent는 5번에서 실행하고, 6번 helper와 6-1 UI가 같은 structured response를 사용한다.


In [ ]:
# 흐름: reminder 타입을 추가하고, 전체 결과 모델의 kind로 분기할 수 있게 한다.
# kind를 먼저 보면 schedule/todo/reminder/unknown 중 어떤 객체를 읽을지 알 수 있다.
class ReminderCreate(BaseModel):
    # 알림 요청은 기준 사건과 몇 분 전인지 숫자를 분리해서 담는다.
    title: str = Field(description="알림 제목")
    related_event: str | None = Field(default=None, description="알림이 연결된 일정이나 사건")
    offset_minutes: int = Field(description="기준 사건 몇 분 전에 알릴지")

class PracticeExtractionResult(BaseModel):
    # reminder가 추가됐으므로 kind 후보와 세부 필드도 함께 늘린다.
    kind: Literal["schedule", "todo", "reminder", "unknown"]
    schedule: ScheduleCreate | None = None
    todo: TodoCreate | None = None
    reminder: ReminderCreate | None = None
    question: str | None = None

practice_extract_agent = create_agent(
    model=make_model(),
    tools=[],
    response_format=PracticeExtractionResult,
    system_prompt=(
        "오늘은 2026-04-23이다. 사용자 요청을 schedule, todo, reminder, unknown 중 하나로 구조화한다. "
        "'N분 전에 알려줘' 같은 요청은 reminder로 분류하고 offset_minutes에는 N을 정수로 넣는다."
    ),
)


## 5. 확장 예제 실행

추가한 `reminder` 타입으로 알림 요청을 구조화한다. 수강생은 `practice_request`를 바꿔보며 Pydantic 객체의 `kind`와 세부 필드가 어떻게 달라지는지 확인한다.


In [ ]:
# 흐름: 실제 요청을 실행하고 structured_response를 꺼내 Pydantic 객체로 확인한다.
# assert는 kind와 세부 필드가 서로 맞는지 검증한다.
practice_request = "발표 30분 전에 알려줘"
practice_result = practice_extract_agent.invoke({"messages": [{"role": "user", "content": practice_request}]})
# 여기서부터는 모델 문장이 아니라 Pydantic 필드를 기준으로 검증한다.
practice_response = practice_result["structured_response"]
show_json(practice_response.model_dump())

assert practice_response.kind == "reminder"
assert practice_response.reminder is not None
assert practice_response.reminder.offset_minutes == 30
print("2주차 확장 예제 실행 통과")


{
  "kind": "reminder",
  "schedule": null,
  "todo": null,
  "reminder": {
    "title": "발표 알림",
    "related_event": null,
    "offset_minutes": 30
  },
  "question": null
}
2주차 확장 예제 실행 통과


## 6. 회고

개념 확인 질문:

1. 자유 문장 답변을 그대로 앱 데이터로 쓰면 어떤 문제가 생길 수 있는가?
2. `kind="group_schedule"`와 `kind="reminder"`일 때 반드시 확인해야 할 필드는 무엇인가?
3. `unknown` 결과는 실패인가, 아니면 안전한 분류인가?

작은 응용 과제: 일정 요청, 할 일 요청, 알림 요청, 애매한 질문을 각각 넣고 `kind`와 세부 객체가 어떻게 달라지는지 표로 정리한다.
